# LUCAS grasslands with EUNIS-lvl2 habitat labels
## Data pre-processing of: 

In [1]:
%load_ext autoreload
%autoreload 2

import os
import io
import base64
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML, Markdown
from matplotlib_venn import venn2

from data_viz import step_plot_distribution, pie_chart_multilabels_proportion, pie_chart_simple, bar_plot_distribution_with_floating_text, bar_plot_habitats_distribution_VS_syntaxons, stack_bars_habitats
from stats import get_basic_stats

In [2]:
def fig_to_img(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    data = base64.b64encode(buf.read()).decode("utf-8")
    return f'<img src="data:image/png;base64,{data}" style="margin-right:20px;">'

def display_plots_side_by_side(figs):
    html = f"""<div style="display:flex; align-items:flex-start;">"""
    for fig in figs:
        html += f'{fig_to_img(fig)}'
    html += '\n</div>'
    display(HTML(html))

In [3]:
ROOT_PATH_OCCURRENCES = 'data/output/csv/'

df_occurrences_18_raw = pd.read_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas18.csv'), low_memory=False)
df_exif_18_raw = pd.read_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas18_exif.csv'), low_memory=False)
df_occurrences_22_raw = pd.read_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas22.csv'), low_memory=False)
df_exif_22_raw = pd.read_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas22_exif.csv'), low_memory=False)

In [4]:
print(f'\033[1m● \x1B[4mOccurences 2018\x1B[0m\033[0m: \n > \x1B[4mShape\x1B[0m: {df_occurrences_18_raw.shape} \n > \x1B[4mColumns\x1B[0m: {df_occurrences_18_raw.columns.tolist()}')
display(df_occurrences_18_raw.head(1))
print(f'\033[1m● \x1B[4mExif 2018\x1B[0m\033[0m: \n > \x1B[4mShape\x1B[0m: {df_exif_18_raw.shape} \n > \x1B[4mColumns\x1B[0m: {df_exif_18_raw.columns.tolist()}')
display(df_exif_18_raw.head(1))
print(f'\033[1m● \x1B[4mOccurences 2022\x1B[0m\033[0m: \n \x1B[4mShape\x1B[0m: {df_occurrences_22_raw.shape} \n \x1B[4mColumns\x1B[0m: {df_occurrences_22_raw.columns.tolist()}')
display(df_occurrences_22_raw.head(1))
print(f'\033[1m● \x1B[4mExif 2022\x1B[0m\033[0m: \n > \x1B[4mShape\x1B[0m: {df_exif_22_raw.shape} \n > \x1B[4mColumns\x1B[0m: {df_exif_22_raw.columns.tolist()}')
display(df_exif_22_raw.head(1))

● Occurences 2018: 
 > Shape: (2622, 128) 
 > Columns: ['fid', 'point_id', 'survey_grass_gps_lat', 'survey_grass_gps_lon', 'survey_grass_gps_ew', 'gps_status', 'distancetothloc', 'point_grass_region', 'point_grass_region_name', 'point_altitude', 'areacode', 'alt_class', 'nuts0', 'lc1', 'survey_date', 'survey_grass_cando', 'survey_grass_trnsct_shift', 'survey_grass_trnsct_shift_dist', 'survey_grass_trnsct_direction', 'survey_grass_trnsct_length', 'survey_grass_slope', 'survey_grass_eunis_habitat', 'survey_grass_orientation', 'survey_grass_site_moisture', 'survey_grass_surface', 'survey_grass_animal_paths', 'survey_grass_fertiliz', 'survey_grass_fertiliz_type', 'survey_grass_grassland_type', 'survey_grass_meadow_growth', 'survey_grass_pasture_cattle', 'survey_grass_pasture_horses', 'survey_grass_pasture_sheep', 'survey_grass_pasture_goats', 'survey_grass_pasture_donkeys', 'survey_grass_pasture_pigs', 'survey_grass_pasture_geese', 'survey_grass_pasture_deer', 'survey_grass_other_type', 's

,fid,point_id,survey_grass_gps_lat,survey_grass_gps_lon,survey_grass_gps_ew,gps_status,distancetothloc,point_grass_region,point_grass_region_name,point_altitude,...,survey_grass_woody_other_perc,survey_grass_woody_dead_perc,survey_grass_herb_layer2_h_cm,survey_grass_herb_layer3_h_cm,survey_grass_herb_layer4_h_cm,survey_grass_herb_layer5_h_cm,survey_grass_pasture_grazing,survey_grass_trnsct_start,nuts3,geom
0,1,26521776,37.19606,-8.860353,2,Original,3.715658,5,Mediterranean - West + Central,31,...,NaN,NaN,5.0,NaN,NaN,NaN,After 1st grazing,On the point,PT150,0101000020E6100000EE073C3080B821C0E2067C7E1899...


● Exif 2018: 
 > Shape: (21421, 65) 
 > Columns: ['sourcefile', 'filesize', 'filetype', 'make', 'model', 'xresolution', 'yresolution', 'resolutionunit', 'exposuretime', 'fnumber', 'datetimeoriginal', 'createdate', 'focallength', 'focallengthin35mmformat', 'imagewidth', 'imageheight', 'fov', 'focallength35efl', 'digitalzoom', 'cameraorientation', 'aspectratio', 'gpsdatestamp', 'gpstimestamp', 'gpsdatetime', 'gpslatituderef', 'gpslongituderef', 'gpsaltituderef', 'gpsspeedref', 'gpsspeed', 'gpsimgdirectionref', 'gpsimgdirection', 'gpsdestbearingref', 'gpsdestbearing', 'gpshpositioningerror', 'gpsaltitude', 'gpslatitude', 'gpslongitude', 'gpsposition', 'gpsversionid', 'gpssatellites', 'gpsmapdatum', 'gpsprocessingmethod', 'gpsstatus', 'gpsmeasuremode', 'gpsdop', 'gpsareainformation', 'gpsdifferential', 'gpstrackref', 'gpstrack', 'usercomment', 'devicemanufacturer', 'devicemodel', 'deviceattributes', 'imagequality', 'advancedscenetype', 'jpegquality', 'TimeStamp', 'digitalzoomratio', 'point

,sourcefile,filesize,filetype,make,model,xresolution,yresolution,resolutionunit,exposuretime,fnumber,...,jpegquality,TimeStamp,digitalzoomratio,point_id,img_letter_group,survey_type,orientation,gpsdestlatituderef,gpsdestlongituderef,gpsdestdistanceref
0,/eos/jeodpp/data/projects/REFOCUS/data/LUCAS20...,306915,JPEG,NaN,NaN,1.0,1.0,0.0,NaN,NaN,...,NaN,NaN,NaN,49343014,M,surveyor,NaN,NaN,NaN,NaN


● Occurences 2022: 
 Shape: (12119, 116) 
 Columns: ['point_id', 'user_id', 'point_nuts0', 'pi_extension', 'point_ex_ante', 'point_lat', 'point_long', 'point_altitude', 'point_copernicus', 'point_grassland', 'grass_region_name', 'point_grass_subregion', 'point_grassland_extended', 'point_erosion', 'point_lf', 'point_soil', 'point_soil_organic', 'point_soil_bulk_0_10', 'point_soil_bulk_10_20', 'point_soil_bulk_20_30', 'point_soil_bio', 'survey_date', 'survey_start_time', 'survey_end_time', 'survey_car_latitude', 'survey_car_longitude', 'survey_gps_lat', 'survey_gps_long', 'survey_gps_altitude', 'survey_gps_prec', 'survey_gps_proj', 'SURVEY_GPS_PROJ.1', 'survey_obs_dist', 'survey_calc_dist', 'survey_obs_type', 'survey_homplot_fills_extwin', 'survey_obs_direct', 'survey_reason_dirchange', 'survey_reason_border_from', 'survey_reason_border_to', 'survey_reason_lf', 'survey_lc1', 'survey_lc1_spec', 'survey_lc1_perc', 'survey_lc2', 'survey_lc2_spec', 'survey_lc2_perc', 'survey_lu1', 'survey_l

,point_id,user_id,point_nuts0,pi_extension,point_ex_ante,point_lat,point_long,point_altitude,point_copernicus,point_grassland,...,survey_grass_richness_spec12,survey_grass_richness_spec7,survey_grass_struc_spec1_perc,survey_grass_struc_spec3_perc,survey_grass_struc_spec5_perc,survey_grass_struc_spec7_perc,survey_grass_struc_spec8_perc,survey_grass_struc_spec10_perc,survey_grass_legume_total_perc,geom_surv_4326
0,26421766,PTSU012,PT,0,0,37.086044,-8.945345,140,1,1,...,0,0,0 %,>3 - 8 %,0 %,0 %,0 %,0 %,0 %,0101000020E61000000000006004E421C000000080038B...


● Exif 2022: 
 > Shape: (49699, 63) 
 > Columns: ['sourcefile', 'filesize', 'filetype', 'resolutionunit', 'xresolution', 'yresolution', 'imagedescription', 'make', 'model', 'orientation', 'exposuretime', 'datetimeoriginal', 'createdate', 'aperturevalue', 'focallength', 'gpslatituderef', 'gpslongituderef', 'gpsaltituderef', 'gpstimestamp', 'gpsdatestamp', 'Country-PrimaryLocationCode', 'Country-PrimaryLocationName', 'headline', 'credit', 'Source', 'imagewidth', 'imageheight', 'aperture', 'gpsaltitude', 'gpsdatetime', 'gpslatitude', 'gpslongitude', 'focallength35efl', 'gpsposition', 'fnumber', 'usercomment', 'digitalzoomratio', 'focallengthin35mmformat', 'gpsversionid', 'fov', 'devicemanufacturer', 'devicemodel', 'deviceattributes', 'gpsspeedref', 'gpsspeed', 'gpsprocessingmethod', 'gpsimgdirectionref', 'gpsimgdirection', 'gpsmapdatum', 'gpsdestbearingref', 'gpsdestbearing', 'gpshpositioningerror', 'warning', 'gpssatellites', 'gpsstatus', 'gpsmeasuremode', 'gpstrackref', 'gpsdestlatitude

,sourcefile,filesize,filetype,resolutionunit,xresolution,yresolution,imagedescription,make,model,orientation,...,gpssatellites,gpsstatus,gpsmeasuremode,gpstrackref,gpsdestlatituderef,gpsdestlongituderef,gpsdestdistanceref,gpsareainformation,aspectratio,point_id
0,/eos/jeodpp/data/projects/REFOCUS/data/LUCAS20...,398602,JPEG,0,1,1,"LUCAS 2022, 42862684, GRASS_Vigour",samsung,SM-G390F,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42862684


In [5]:
print(f'Nb of unique point_id in occurrence file for 2018: \033[1;34m{df_occurrences_18_raw['point_id'].nunique()}\x1B[0m')
print(f'Nb of unique point_id in exif file for 2022: \033[1;34m{df_exif_18_raw['point_id'].nunique()}\x1B[0m')

print(f'Nb of unique point_id in occurrence file for 2018: \033[1;34m{df_occurrences_22_raw['point_id'].nunique()}\x1B[0m')
print(f'Nb of unique point_id in exif file for 2022: \033[1;34m{df_exif_22_raw['point_id'].nunique()}\x1B[0m')

Nb of unique point_id in occurrence file for 2018: 2622
Nb of unique point_id in exif file for 2022: 2472
Nb of unique point_id in occurrence file for 2018: 12119
Nb of unique point_id in exif file for 2022: 9400


## Cleaning CSVs

In [6]:
def clean_file_paths(s, prefix):
    """Replaces the folder path of the JRC servers to our local file structure."""
    suffix = '/'.join(s.split('/')[-4:])
    return os.path.join(prefix, suffix)

In [10]:
df_exif_18 = df_exif_18_raw.copy()
df_exif_22 = df_exif_22_raw.copy()
prefix_18 = 'data/output/data/LUCAS2018_Grasslands/'
prefix_22 = 'data/output/data/LUCAS2022_Grasslands/'
df_exif_18['sourcefile'] = df_exif_18_raw['sourcefile'].apply(clean_file_paths, prefix=prefix_18)
df_exif_22['sourcefile'] = df_exif_22_raw['sourcefile'].apply(clean_file_paths, prefix=prefix_22)

print(f"Filepath from LUCAS18 changed from \033[1;32m'{'/'.join(df_exif_18_raw['sourcefile'].iloc[0].split('/')[:-4])}\033[1;34m{'/'.join(df_exif_18_raw['sourcefile'].iloc[0].split('/')[-4:])}'\x1B[0m"
      f"to \033[1;32m'{prefix_18}\033[1;34m{df_exif_18['sourcefile'].iloc[0].split(prefix_18)[1]}\x1B[0m")
print(f"Filepath from LUCAS22 changed from \033[1;32m'{'/'.join(df_exif_22_raw['sourcefile'].iloc[0].split('/')[:-4])}/\033[1;34m{'/'.join(df_exif_22_raw['sourcefile'].iloc[0].split('/')[-4:])}'\x1B[0m"
      f"to \033[1;32m'{prefix_22}\033[1;34m{df_exif_22['sourcefile'].iloc[0].split(prefix_22)[1]}\x1B[0m")

Filepath from LUCAS18 changed from '/eos/jeodpp/data/projects/REFOCUS/data/LUCAS2018_Grassland/images/surveyor_photo/CZ/493/430/49343014M.jpg'to 'data/output/data/LUCAS2018_Grasslands/CZ/493/430/49343014M.jpg
Filepath from LUCAS22 changed from '/eos/jeodpp/data/projects/REFOCUS/data/LUCAS2022_photos/Grassland_photos//grasslandPhotos/AT/428/626/202242862684GRASS_Vigour.jpg'to 'data/output/data/LUCAS2022_Grasslands/AT/428/626/202242862684GRASS_Vigour.jpg


## Verifying files

Let's verify if all the listed photos are properly downloaded.

In [11]:
def verify_image_integrity(fp):
    exists = os.path.exists(fp)
    weight = os.path.getsize(fp)
    empty = weight < 100
    return exists, empty, fp

def verify_images_integrity(df, fp_col):
    verifs = {'valid': [],
              'invalid': {'reason': [],
                          'point_id': [],
                          'fp': []}}
    for rowi, row in tqdm(df.iterrows(), total=len(df)):
        exists, empty, fp = verify_image_integrity(row[fp_col])
        if exists and not empty:  # A ∩ B_bar
            verifs['valid'].append(fp)
        if not exists or empty:  # A_bar ∪ B
            verifs['invalid']['fp'].append(fp)
            verifs['invalid']['point_id'].append(row['point_id'])
            if not exists:
                verifs['invalid']['reason'].apend('File does not exist')
            if empty:
                verifs['invalid']['reason'].append('File is empty (no data)')
    all_imgs_valid =  len(verifs['valid']) == len(df)
    return all_imgs_valid, verifs

In [12]:
print(f'Verifying LUCAS 2018 grasslands habitat dataset...')
all_imgs_valid18, verifs18 = verify_images_integrity(df_exif_18, 'sourcefile')
print(f'Verifying LUCAS 2022 grasslands habitat dataset...')
all_imgs_valid22, verifs22 = verify_images_integrity(df_exif_22, 'sourcefile')

if all_imgs_valid18 and all_imgs_valid22:
    print(f'All images exist downloaded and verified !')
else:
    print("Some images were not valid.\n- 2018:")
    for reason, point_id, fp in zip(verifs18["invalid"]['reason'], verifs18["invalid"]['point_id'], verifs18["invalid"]['fp']):
        print(f'{reason}: {point_id}, {fp}')
    print("- 2022: ")
    for reason, point_id, fp in zip(verifs22["invalid"]['reason'], verifs22["invalid"]['point_id'], verifs22["invalid"]['fp']):
        print(f'{reason}: {point_id}, {fp}')

Verifying LUCAS 2018 grasslands habitat dataset...


100%|██████████████████████████████████| 21421/21421 [00:00<00:00, 57091.05it/s]


Verifying LUCAS 2022 grasslands habitat dataset...


100%|██████████████████████████████████| 49699/49699 [00:00<00:00, 56557.88it/s]

Some images were not valid.
- 2018:
File is empty (no data): 36582596, data/output/data/LUCAS2018_Grasslands/FR/365/825/36582596U.jpg
File is empty (no data): 39962408, data/output/data/LUCAS2018_Grasslands/FR/399/624/39962408M.jpg
File is empty (no data): 44382522, data/output/data/LUCAS2018_Grasslands/IT/443/825/44382522R.jpg
File is empty (no data): 47763500, data/output/data/LUCAS2018_Grasslands/PL/477/635/47763500R.jpg
File is empty (no data): 27841918, data/output/data/LUCAS2018_Grasslands/PT/278/419/27841918Q.jpg
File is empty (no data): 28201984, data/output/data/LUCAS2018_Grasslands/PT/282/019/28201984M.jpg
File is empty (no data): 53162586, data/output/data/LUCAS2018_Grasslands/RO/531/625/53162586Q.jpg
File is empty (no data): 54782658, data/output/data/LUCAS2018_Grasslands/RO/547/826/54782658Q.jpg
File is empty (no data): 56762572, data/output/data/LUCAS2018_Grasslands/RO/567/625/56762572M.jpg
File is empty (no data): 45543610, data/output/data/LUCAS2018_Grasslands/SE/455/43

In [13]:
for verifs, df_exif, year in zip([verifs18, verifs22], [df_exif_18, df_exif_22], ['2018', '2022']):
    n_occur = df_exif.shape[0]
    for fp, point_id in zip(verifs["invalid"]['fp'], verifs["invalid"]['point_id']):
        df_exif = df_exif.drop(df_exif[df_exif['sourcefile'] == fp].index)
    match year:
        case '2018':
            df_exif_18_verif = df_exif.copy()
        case '2022':
            df_exif_22_verif = df_exif.copy()
    delta = df_exif.shape[0] - n_occur
    print(f'LUCAS_{year} nb of occurrences after validity check: {n_occur} ->  {df_exif.shape[0]} ({delta} invalid files removed)')

LUCAS_2018 nb of occurrences after validity check: 21421 ->  21411 (-10 invalid files removed)
LUCAS_2022 nb of occurrences after validity check: 49699 ->  49699 (0 invalid files removed)


### Verify truncated files (/!\ resource expensive !)

In [27]:
paths_to_remove

{'data/output/data/LUCAS2018_Grasslands/BG/538/022/53802208Q.jpg'}

In [31]:
for i in df_exif_18_verif[df_exif_18_verif['point_id'] == 53802208]['sourcefile']:
    print(i)

data/output/data/LUCAS2018_Grasslands/BG/538/022/53802208E.jpg
data/output/data/LUCAS2018_Grasslands/BG/538/022/53802208M.jpg
data/output/data/LUCAS2018_Grasslands/BG/538/022/53802208N.jpg
data/output/data/LUCAS2018_Grasslands/BG/538/022/53802208P.jpg
data/output/data/LUCAS2018_Grasslands/BG/538/022/53802208R.jpg
data/output/data/LUCAS2018_Grasslands/BG/538/022/53802208S.jpg
data/output/data/LUCAS2018_Grasslands/BG/538/022/53802208U.jpg
data/output/data/LUCAS2018_Grasslands/BG/538/022/53802208W.jpg


In [26]:
print('Verifying truncated images...')
# os.system('python check_corrupted_images_multiprocessing.py --root data/output/data/')

with open("truncated_images.txt") as f:
    paths_to_remove = f.readlines()

paths_to_remove = set(paths_to_remove)  # Faster lookups
def remove_paths(cell):
    paths = [p for p in cell.split(";") if p not in paths_to_remove]
    return ";".join(paths)

if len(paths_to_remove) > 0:
    print(f'Found {len(paths_to_remove)} truncated images. Removed them from exif dataframes.')

df_exif_18_verif["sourcefile"] = df_exif_18_verif["sourcefile"].apply(remove_paths)
df_exif_22_verif["sourcefile"] = df_exif_22_verif["sourcefile"].apply(remove_paths)

# Remove rows that became empty
df_exif_18_verif = df_exif_18_verif[df_exif_18_verif["sourcefile"] != ""].copy()
df_exif_22_verif = df_exif_22_verif[df_exif_22_verif["sourcefile"] != ""].copy()

Verifying truncated images...
Found 1 truncated images. Removed them from exif dataframes.


In [18]:
uhabitats_18 = df_occurrences_18_raw[df_occurrences_18_raw['point_id'].isin(df_exif_18_verif['point_id'])]['survey_grass_eunis_habitat'].nunique()
uhabitats_22 = df_occurrences_22_raw[df_occurrences_22_raw['point_id'].isin(df_exif_22_verif['point_id'])]['survey_grass_eunis_habitat_type'].nunique()

# print(f'Nb of images in LUCAS18 grasslands: {df_exif_18_verif.shape[0]}, in LUCAS22 grasslands: {df_exif_22.shape[0]}')
# print(f'Nb of plots in LUCAS18 grasslands: {df_exif_18_verif['point_id'].nunique()}, in LUCAS22 grasslands: {df_exif_22['point_id'].nunique()}')
# print(f'Nb of unique EUNIS-lvl2 habitats in LUCAS18 grasslands: {uhabitats_18}, in LUCAS22 grasslands: {uhabitats_22}')

fmt = lambda x: f"{x:,}".replace(",", " ")
table = f"""
| Metric | LUCAS18 grasslands | LUCAS22 grasslands |
|---|---:|---:|
| Nb of images | {fmt(df_exif_18_verif.shape[0])} | {fmt(df_exif_22_verif.shape[0])} |
| Nb of plots | {fmt(df_exif_18_verif['point_id'].nunique())} | {fmt(df_exif_22_verif['point_id'].nunique())} |
| Nb of unique EUNIS-lvl2 habitats | {fmt(uhabitats_18)} | {fmt(uhabitats_22)} |
"""
display(Markdown(table))


| Metric | LUCAS18 grasslands | LUCAS22 grasslands |
|---|---:|---:|
| Nb of images | 21 410 | 49 699 |
| Nb of plots | 2 472 | 9 400 |
| Nb of unique EUNIS-lvl2 habitats | 28 | 24 |


## Merging and harmonizing files

Files 2018 and 2022 havedifferent column names and use EUNIS 2007 codes. For harmonizaing purposes, we need to align the columns numbers and map the EUNIS lvl2 codes to the 2023 nomenclature, used in our FloraVeg experiments and in GeoPl@ntNet. **LUCAS grasslands labels are multi-class, meaning only 1 among k labels per site.**

**NOTE**: the EUNIS conversion table from nomenclature 2007 to 2023 highlights that some 2007 codes may be mapped to multiple 2023 codes, of multiple levels. For e.g., _A2.5_ maybe matched with _MA2_ or any higher level with _MA2_ base. **Since LUCAS grasslands labels are only supposed to be LVL-2, we will assume only LVL-2 mapping are relevant.**

In [19]:
def map_eunis_code_2007_to_2023(
    df_to_map: pd.DataFrame,
    fp_eunis_map: str = '../EUNIS_habitats/EUNIS-habitats-2025-10-03.csv',
    restrict_mapping_lvl: list[int] = [2],
    col_habitat_to_map: str = 'habitats_code_lvl2',
    verbose: bool = False
):
    df_eunis_map = pd.read_csv(fp_eunis_map)
    df_eunis_map.dropna(subset='EUNIS 2007 code', inplace=True)
    df_eunis_map = df_eunis_map[df_eunis_map['Hierarchy'].isin([f'Level {lvl}' for lvl in restrict_mapping_lvl])]
        
    eunis_map = {}  # keys = 2007, values = 2023
    for codes_2007, code_2023 in zip(df_eunis_map['EUNIS 2007 code'].values, df_eunis_map['EUNIS 2023 code'].values):
        for code_2007 in codes_2007.split(';'):
            if code_2007 in eunis_map.keys():
                eunis_map[code_2007] = f'{eunis_map[code_2007]}; {code_2023}'.strip()
            else:
                eunis_map[code_2007] = str(code_2023).strip()

    uh_counts = df_18_22['habitats_code_lvl2'].value_counts()
    unmappable_habitats = {k: v for k, v in uh_counts.items() if k not in eunis_map.keys()}

    if verbose:
        print('EUNIS_map: ', eunis_map)
        display('unmappable_habitats: ', unmappable_habitats)

    if len(unmappable_habitats) > 0:
        print(f'\n[INFO]: The following LVL-2 habitats from LUCAS grasslands are NOT mappable to a EUNIS2023 code: {list(unmappable_habitats.keys())} ({sum(unmappable_habitats.values())} photos ({100*sum(unmappable_habitats.values())/len(df_18_22):.2f}%)).\n')

    df_to_map = df_to_map[~df_to_map['habitats_code_lvl2'].isin(unmappable_habitats)]
    df_to_map[f'{col_habitat_to_map}_2023'] = df_to_map[col_habitat_to_map].map(eunis_map)
    return df_to_map, eunis_map

In [20]:
columns18_to_keep = ['point_id', 'survey_grass_gps_lat', 'survey_grass_gps_lon', 'gps_status', 'point_altitude',  'survey_grass_eunis_habitat', 'sourcefile']
columns22_to_keep = ['point_id', 'point_lat', 'point_long', 'point_altitude',  'survey_grass_eunis_habitat_type', 'sourcefile']

# 1. Merge exif et occurrences for each year
df_18 = df_occurrences_18_raw.merge(
    df_exif_18_verif,
    on="point_id",
    how="inner"
)
df_18 = df_18.loc[:, columns18_to_keep]

df_22 = df_occurrences_22_raw.merge(
    df_exif_22_verif,
    on="point_id",
    how="inner"
)
df_22 = df_22.loc[:, columns22_to_keep]

display(df_18.sample())
display(df_22.sample())

,point_id,survey_grass_gps_lat,survey_grass_gps_lon,gps_status,point_altitude,survey_grass_eunis_habitat,sourcefile
10530,45342676,47.16125,12.80756,Original,1201,E4,data/output/data/LUCAS2018_Grasslands/AT/453/4...


,point_id,point_lat,point_long,point_altitude,survey_grass_eunis_habitat_type,sourcefile
27160,46122132,42.23148,13.514442,1257,E2 - Mesic grasslands,data/output/data/LUCAS2022_Grasslands/IT/461/2...


In [21]:
# 2. Harmonize & concat both merged years
columns18_rename = {'survey_grass_gps_lat': 'lat', 'survey_grass_gps_lon': 'lon', 'point_altitude': 'altitude', 'survey_grass_eunis_habitat': 'habitats_code_lvl2', 'sourcefile': 'filepath'}
columns22_rename = {'point_lat': 'lat', 'point_long': 'lon', 'point_altitude': 'altitude', 'survey_grass_eunis_habitat_type': 'habitats_code_lvl2', 'sourcefile': 'filepath'}

## 2018
df_18.rename(columns=columns18_rename, inplace=True)
df_18['year'] = ['2018']*len(df_18)
df_18 = df_18.dropna(subset=['point_id', 'lon', 'lat', 'habitats_code_lvl2', 'filepath'])
df_18['habitats_code_lvl2'] = df_18['habitats_code_lvl2'].apply(lambda x: x.strip())
df_18 = df_18[df_18['habitats_code_lvl2'] != 'XX']
df_18.rename(columns={'filepath': 'filepath_2018'}, inplace=True)

## 2022
df_22.rename(columns=columns22_rename, inplace=True)
df_22['year'] = ['2022']*len(df_22)
df_22 = df_22.dropna(subset=['point_id', 'lon', 'lat', 'habitats_code_lvl2', 'filepath'])
df_22['habitats_code_lvl2'] = df_22['habitats_code_lvl2'].apply(lambda x: x.split(' - ')[0])
df_22['habitats_code_lvl2'] = df_22['habitats_code_lvl2'].apply(lambda x: x.strip())
df_22 = df_22[df_22['habitats_code_lvl2'] != 'XX']
df_22.rename(columns={'filepath': 'filepath_2022'}, inplace=True)

## Merge
df_18_22 = pd.concat([df_18, df_22], ignore_index=True)
df_18_22['filepath_2018'] = df_18_22['filepath_2018'].fillna('')
df_18_22['filepath_2022'] = df_18_22['filepath_2022'].fillna('')

# 3. Map EUNIS habitat codes from 2007 nomenclature to 2023's
df_18_22, df_eunis_map = map_eunis_code_2007_to_2023(df_18_22)

# 4. Verify stats
## Re-compute stats
n_img_18 = df_18.shape[0]
n_img_22 = df_22.shape[0]
n_img_18_22 = df_18_22.shape[0]
n_plots_18 = df_18['point_id'].nunique()
n_plots_22 = df_22['point_id'].nunique()
n_plots_18_22 = df_18_22['point_id'].nunique()
uh_18 = df_18['habitats_code_lvl2'].unique()
uh_22 = df_22['habitats_code_lvl2'].unique()
uh_18_22 = df_18_22['habitats_code_lvl2'].unique()

## Display
fmt = lambda x: f"{x:,}".replace(",", " ")
table = f"""
| Metric | LUCAS18+22 grasslands | LUCAS18 grasslands | LUCAS22 grasslands
|---|---:|---:|---:|
| Nb of images | {fmt(n_img_18_22)} |{fmt(n_img_18)} | {fmt(n_img_22)} |
| Nb of plots | {fmt(n_plots_18_22)} |{fmt(n_plots_18)} | {fmt(n_plots_22)} |
| Nb of unique EUNIS-lvl2 habitats | {fmt(len(uh_18_22))} |{fmt(len(uh_18))} | {fmt(len(uh_22))} |
"""
display(Markdown(table))

# 4. Save cleaned & harmonized grasslands file
df_18_22.to_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas_grasslands_habitats.csv'), sep=',', index=False)


[INFO]: The following LVL-2 habitats from LUCAS grasslands are NOT mappable to a EUNIS2023 code: ['G4', 'D5', 'D6'] (292 photos (0.41%)).




| Metric | LUCAS18+22 grasslands | LUCAS18 grasslands | LUCAS22 grasslands
|---|---:|---:|---:|
| Nb of images | 70 335 |21 343 | 49 284 |
| Nb of plots | 11 339 |2 464 | 9 322 |
| Nb of unique EUNIS-lvl2 habitats | 25 |27 | 23 |


## Site-aggregation
Make an aggregated version of the dataset by sites for habitat-prediction pipeline compatibility

In [22]:
def _make_agg_dict(df):
    def _agg(x):
        x = ';'.join(x.astype(str))
        x = np.unique(x.split(';'))
        x = x[x != '']
        x = ';'.join(x)
        return x
        
    agg_dict = {}
    for col in df.columns:
        if col in ['point_id']:
                pass
        elif col in [
            'filepath_2018', 'filepath_2022',
            'habitats_code', 'habitats_code_lvl1', 'habitats_code_lvl2', 'habitats_code_lvl3', 'habitats_code_lvl4',
            'habitats_code_ID', 'habitats_code_ID_lvl1', 'habitats_code_ID_lvl2', 'habitats_code_ID_lvl3', 'habitats_code_ID_lvl4',
            'habitats_code_lvl2_2023',
        ]:
            agg_dict[col] = lambda x: _agg(x)  # lambda x: ';'.join(x.astype(str))
        else:
            agg_dict[col] = 'first'
    return agg_dict

def count_u_habitats_multilabel(df):
    u_habitats = {}
    for rowi, row in df.iterrows():
        ks = row['habitats_code_lvl2_2023'].split(';')
        for k in ks:
            u_habitats[k] = u_habitats.get(k, 0) + 1
    return u_habitats

In [23]:
df_gb_18 = df_18_22[df_18_22['year'] == '2018']
df_gb_22 = df_18_22[df_18_22['year'] == '2022']
df_gb_18 = df_gb_18.groupby('point_id', as_index=False).agg(_make_agg_dict(df_gb_18))
df_gb_22 = df_gb_22.groupby('point_id', as_index=False).agg(_make_agg_dict(df_gb_22))
df_18_22_gb = df_18_22.groupby('point_id', as_index=False).agg(_make_agg_dict(df_18_22))

In [24]:
df_gb_18_multilabel = df_gb_18[df_gb_18['habitats_code_lvl2'].str.contains(';')]
print(f'In 2018, there are {len(df_gb_18_multilabel)}/{len(df_gb_18)} sites ({100*len(df_gb_18_multilabel)/len(df_gb_18):.2f}%) with multi-label labels.')
display(df_gb_18_multilabel)

df_gb_22_multilabel = df_gb_22[df_gb_22['habitats_code_lvl2'].str.contains(';')]
print(f'In 2022, there are {len(df_gb_22_multilabel)}/{len(df_gb_22)} sites ({100*len(df_gb_22_multilabel)/len(df_gb_22):.2f}%) with multi-label labels.')
display(df_gb_18_multilabel)

df_18_22_gb_multilabel = df_18_22_gb[df_18_22_gb['habitats_code_lvl2'].str.contains(';')]
print(f'There are {len(df_18_22_gb_multilabel)}/{len(df_18_22_gb)} sites ({100*len(df_18_22_gb_multilabel)/len(df_18_22_gb):.2f}%) with multi-label labels (1 for each year).')
display(df_18_22_gb_multilabel.sample(1))

df_18_22_gb_multiclass = df_18_22_gb[~df_18_22_gb['point_id'].isin(df_18_22_gb_multilabel['point_id'])]

assert len(df_18_22_gb) == (len(df_18_22_gb_multilabel) + len(df_18_22_gb_multiclass))

fmt = lambda x: f"{x:,}".replace(",", " ")
table = f"""
| Metric | LUCAS18+22 | LUCAS18+22 multilabel | LUCAS18+22 multiclass | LUCAS18 | LUCAS22
|---|---:|---:|---:|---:|---:|
| Nb of plots | {fmt(len(df_18_22_gb))} | {fmt(len(df_18_22_gb_multilabel))} | {fmt(len(df_18_22_gb_multiclass))} |{fmt(n_plots_18)} | {fmt(n_plots_22)} |
| Nb of unique EUNIS-lvl2 habitats | {fmt(len(count_u_habitats_multilabel(df_18_22_gb)))} | {fmt(len(count_u_habitats_multilabel(df_18_22_gb_multilabel)))} | {fmt(df_18_22_gb_multiclass['habitats_code_lvl2_2023'].nunique())} |{fmt(len(uh_18))} | {fmt(len(uh_22))} |
|Description|Both multilabel and multiclass samples|Multilabel samples ONLY|Multiclass samples ONLY|All samples are multiclass|All samples are multiclass|
"""
display(Markdown(table))

In 2018, there are 0/2458 sites (0.00%) with multi-label labels.


,point_id,lat,lon,gps_status,altitude,habitats_code_lvl2,filepath_2018,year,filepath_2022,habitats_code_lvl2_2023


In 2022, there are 0/9273 sites (0.00%) with multi-label labels.


,point_id,lat,lon,gps_status,altitude,habitats_code_lvl2,filepath_2018,year,filepath_2022,habitats_code_lvl2_2023


There are 146/11339 sites (1.29%) with multi-label labels (1 for each year).


,point_id,lat,lon,gps_status,altitude,habitats_code_lvl2,filepath_2018,year,filepath_2022,habitats_code_lvl2_2023
10795,55462142,41.304184,24.670672,Original,1229,E2;E7,data/output/data/LUCAS2018_Grasslands/EL/554/6...,2018,data/output/data/LUCAS2022_Grasslands/EL/554/6...,R2;R7



| Metric | LUCAS18+22 | LUCAS18+22 multilabel | LUCAS18+22 multiclass | LUCAS18 | LUCAS22
|---|---:|---:|---:|---:|---:|
| Nb of plots | 11 339 | 146 | 11 193 |2 464 | 9 322 |
| Nb of unique EUNIS-lvl2 habitats | 25 | 18 | 25 |27 | 23 |
|Description|Both multilabel and multiclass samples|Multilabel samples ONLY|Multiclass samples ONLY|All samples are multiclass|All samples are multiclass|


In [25]:
df_gb_18.to_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas_grasslands_habitats_groupedby_multiclass_2018.csv'), sep=',', index=False)
df_gb_22.to_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas_grasslands_habitats_groupedby_multiclass_2022.csv'), sep=',', index=False)

df_18_22_gb_multiclass.to_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas_grasslands_habitats_groupbedby_multiclass.csv'), sep=',', index=False)
df_18_22_gb.to_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas_grasslands_habitats_groupedby_multilabel.csv'), sep=',', index=False)

---
Examples of multi-label: label has been revised between 2018 and 2022 for identical survey sites.

`/home/tlarcher/Documents/Pl@ntNet/git/malpolon/examples/custom_train/multiscale_jrc/dataset/habitats_gpn25_eva/LUCAS_habitats/data/output/data/LUCAS2018_Grasslands/ES/338/817/33881756Q.jpg` -> "S3", "Temperate and Mediterranean-montane scrub"
`/home/tlarcher/Documents/Pl@ntNet/git/malpolon/examples/custom_train/multiscale_jrc/dataset/habitats_gpn25_eva/LUCAS_habitats/data/output/data/LUCAS2022_Grasslands/ES/338/817/202233881756GRASS_TransectEnd.jpg` -> "V1", "Arable land and market gardens"

`/home/tlarcher/Documents/Pl@ntNet/git/malpolon/examples/custom_train/multiscale_jrc/dataset/habitats_gpn25_eva/LUCAS_habitats/data/output/data/LUCAS2018_Grasslands/ES/311/023/31102306N.jpg` -> "R7", "Sparsely wooded grasslands"
`/home/tlarcher/Documents/Pl@ntNet/git/malpolon/examples/custom_train/multiscale_jrc/dataset/habitats_gpn25_eva/LUCAS_habitats/data/output/data/LUCAS2022_Grasslands/ES/311/023/202231102306GRASS_Posy.jpg` -> "T1", "Broadleaved deciduous forests"

`/home/tlarcher/Documents/Pl@ntNet/git/malpolon/examples/custom_train/multiscale_jrc/dataset/habitats_gpn25_eva/LUCAS_habitats/data/output/data/LUCAS2018_Grasslands/HR/477/225/47722568U.jpg` -> "R1", "Dry grasslands"
`/home/tlarcher/Documents/Pl@ntNet/git/malpolon/examples/custom_train/multiscale_jrc/dataset/habitats_gpn25_eva/LUCAS_habitats/data/output/data/LUCAS2022_Grasslands/HR/477/225/202247722568GRASS_TransectEnd.jpg` -> "R5", "Woodland fringes and clearings and tall forb stands"